# Writing Workers

In the previous example you wrote a worker only using Tierkreis base functionality.
One of the main benefits of using Tierkreis is that you can easiliy transform existing code and gain the benefits of Tierkeis, e.g., checkpointing and repeteability.

In this example we're going to look at a common task and transform it into a graph.
We're going to use [pytket](https://docs.quantinuum.com/tket/api-docs/) to:
1. Define a symbolic circuit
2. Substitute its symbolic parameters at runtime
3. Compile and execute the circuit


## Prerequisite

If you haven't done so, set up a Tierkreis project and install pytket

```bash
uv init
uv add tierkreis pytket pytket-qiskit sympy
uv run tkr project init
```

## Setting up the worker

Using the the Tierkreis cli is the easiest the way to set up a worker:

```bash
uv run tkr init worker -n pytket_example_worker
uv sync --all-extras
```

Which will set up the packages in a convenient way for you.
```{info}
If you don't want to use the cli, you need a way to provide the worker API to your graph construction.
You can find more information [here](../worker/index.md)
```

There are two  files which you need to care about which are:
- `impl.py` here you will implement your workers functionality
- `api.py` is an autogenerated file which contains the task definitions we will use in the graph.

In `impl.py` you will find the following:

```python
worker = Worker("pytket_example_worker")

@worker.task()
def your_worker_task(value: int) -> int:
    return value
```

As you can see, generating a task for Tierkeis simply means adding `@worker.task()` to a function.
We're going to use the same pattern to expose `pytket`s functionality to in our new worker.

## Defining the tasks

For the following assume we're going to use a simple quantum circuits with three symbols $A,B,C$.


In [ ]:
from pytket.circuit import Circuit, fresh_symbol


a = fresh_symbol("a")
b = fresh_symbol("b")
c = fresh_symbol("c")
circ = Circuit(3)
circ.Rz(a, 0)
circ.Rz(b, 0)
circ.Rz(c, 0)
circ.measure_all()

Using plain `pytket` you could know substitue the symbles like so:

In [ ]:
from sympy import Symbol
circ.symbol_substitution({Symbol("a"): -1, Symbol("b"): 0, Symbol("c"): 1})

writing this as a worker tast means wrapping it with a task function.
```{important}
Use type hints so that Tierkreis can validate the task during construction.
Since values are represnted by edges, we need a return value, its not sufficient to mutate the state.
```

In [ ]:
# This is added for validity of the example
from tierkreis import Worker
worker = Worker("pytket_example_worker")

@worker.task()
def substitute(circuit: Circuit, a: float, b: float, c: float) -> Circuit:
    circuit.symbol_substitution({Symbol("a"): a, Symbol("b"): b, Symbol("c"): c})
    return circuit

Similarly we can now define other tasks using `pytket` and and `@worker.task()`:

For compilation, e.g., optimizing phase gadgets:

In [ ]:
from pytket.transform import Transform

@worker.task()
def optimise(circuit: Circuit) -> Circuit:

    Transform.OptimisePhaseGadgets().apply(circuit)
    return circuit


And simulation, e.g. using on an aer simulator using `pytket-qiskit`

In [ ]:
from pytket.backends.backendresult import BackendResult
from pytket.extensions.qiskit import AerBackend

@worker.task()
def simulate(circuit: Circuit) -> BackendResult:
    backend = AerBackend()
    return backend.run_circuit(circuit, n_shots=1000)


## Generating stubs

To generate the APIs for all workers, you can use the cli with
```bash
uv run tkr init stubs
```

Dpending on your development environment it might be necessary to resatart your language server or `uv sync --all-extras` to pick up the update changes.

## Using the tasks

Now you can use the newly declared tasks in a graph similar to how you used the `builtin` functionality.
You have to import the task API from the worker first which you then can use with a task node.
First we declare the graph

In [ ]:
from typing import NamedTuple
from tierkreis.builder import GraphBuilder
from tierkreis.controller.data.models import TKR

class PytketInputs(NamedTuple):
    circuit: TKR[Circuit]
    a: TKR[float]
    b: TKR[float]
    c: TKR[float]

graph = GraphBuilder(PytketInputs, TKR[BackendResult])


and then add the tasks:

In [ ]:
from pytket_example_worker import substitute, optimisie, simulate

substituted = graph.task(substitute(graph.inputs))
optimized = graph.task(optimise(substituted))
result = graph.task(simulate(optimized))
graph.outputs(result)